# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divyanshu10045/flyrank-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

DRAFT

This one does not build a model. ML-09 built it and audited it; this turns that model into a
reading order — what a person should open first, what the order is allowed to be used for, and
where it stops being true. Everything numeric below is printed by the cell next to it.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

DRAFT

One model, fitted once on the whole pool, orders the pool by how likely a page is to be one of
the material declines in March. The top of that order is the queue.

I fitted exactly one model, and it is ML-09's model with its settings intact: the same five
features, the same degree-2 logistic fitter, and `L2 = 1.0`. That penalty is not a new choice I
made this week — it is the value ML-09's inner grouped CV picked in all six outer folds, chosen
without ever touching a fold's held-out clients. I deliberately did **not** reuse ML-08's shipped
`0.0001`, because ML-09 showed that number was tuned on the very holdout it then reported.

I dropped ML-09's probe feature. `trend_pct_win` is not in the scoring model; adding it made the
honest number worse, not better. I still read it, but only to describe a row, never to score it.

**Actions — three, not five.** My card lists `refresh / expand / protect / prune / monitor`, but
ML-07 already made this call and wrote down why: it dropped `expand` because the word-count idea
behind it came from a field it never tested, and it dropped `prune` because nothing in its evidence
says a page earning 200+ prior impressions should be removed. "Picking five actions would have been
decoration." I am not going to un-drop them in a week when I ran no new test to justify them.

- `refresh` — in the top 50 and already sliding. Open these first; ground is being lost now.
- `protect` — in the top 50 and not sliding. Real volume, currently holding. Hands off.
- `monitor` — everything else. No action this week.

The cut at 50 is not a discovery, it is a resourcing decision — roughly one afternoon of looking
at pages. I picked the number ML-09 actually measured at that depth (P@50), so the operating point
is one I have evidence for rather than one I invented.

**Read this before you use the queue: it is close to a volume ranking, and that has a consequence
I did not expect.** Ranking by my score and ranking by raw prior volume tie at depth 50 — both
0.360 in-sample — and the score correlates 0.80 with prior volume. So the top of this queue is
"the largest pages in the pool", in a repackaged form. Every row above the cut sits in my top volume
band, and only 191 pool rows are that large. Meanwhile the middle band holds **52.3% of all
material declines** and receives **zero** slots in the top 50 — the first row from it appears at
rank 9,741. Even its own best 50 scores the same 0.360 the whole-pool top 50 does.

I could have hidden that by re-weighting the features until the middle band surfaced. I am not
doing that. Re-weighting on a single label is precisely how ML-08 ended up with a penalty tuned on
its own holdout, and it is how a queue like this gets a number that flatters it and a reviewer who
trusts it. This is a real limitation of a one-window volume-driven model, and section 4 has the
trigger for when it should be revisited.

**Why the reason codes look the way they do.** ML-09 shuffled each feature inside held-out blocks
and measured what broke. Prior volume cost 0.234 of P@50 when scrambled. Every other feature cost
between −0.018 and +0.035 — noise, and `ctr_prior_pct` came out negative, meaning scrambling it
*mildly helped*. So a reason code like "low CTR, refresh this" would be a story I invented. My
codes only name the two things that are real: the page's prior-volume band, and whether it is
already sliding. When I say a reason code is a volume-band fact, that is literally the model's
main input, and I would rather be short and true than long and decorative.


In [1]:
# ---- setup: rebuild the exact ML-04/07/08/09 frame from the same aggregates ----
# Reader note: pyarrow's Parquet DLL is blocked by Application Control on this box.
# fastparquet reads the same files, so the reader is the only thing that changed since ML-08.
import os, json
import warnings
from pathlib import Path

# Third-party progress-bar chatter (tqdm, pulled in by huggingface_hub) lands in the captured
# output and drags a documentation URL into the notebook. It says nothing about this analysis.
# tqdm emits its warning at IMPORT time, so the filter has to go in before the import below.
warnings.filterwarnings("ignore", message=r".*IProgress.*")
warnings.filterwarnings("ignore", message=r".*ipywidgets.*")

import numpy as np, pandas as pd
import fastparquet
from huggingface_hub import HfFileSystem
from dotenv import load_dotenv
from scipy.optimize import minimize

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"work"/"notebooks").is_dir() and (p/".git").exists()), Path.cwd())
os.chdir(REPO)
load_dotenv(REPO/".env")
fs = HfFileSystem(token=os.environ.get("HF_TOKEN") or None)

WAREHOUSE = "datasets/FlyRank/internship-warehouse"
FACT = WAREHOUSE + "/fact_content_daily_performance"
KEYS = ["client_hash_id", "content_hash_id"]

FEATURE_START, FEATURE_END = "2026-01-01", "2026-02-28"
LABEL_START,   LABEL_END   = "2026-03-01", "2026-03-31"
DECISION_POINT = LABEL_START
PRIOR_DAYS, LABEL_DAYS = 59, 31
DAYS_IN = {"2026-01": 31, "2026-02": 28, "2026-03": 31}
POOL_FLOOR, VOLUME_FLOOR, DROP_FLOOR = 200, 500, -20
DOWN_FLOOR, TOP_TEN = -20.0, 10
FEATURE_MONTHS, LABEL_MONTHS = ("2026-01", "2026-02"), ("2026-03",)
DAILY_COLS = ["report_date", "client_hash_id", "content_hash_id",
              "gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_data_available"]
ENGINE = "fastparquet"

# Same one-row-per-(client, content) aggregates ML-09 read. Reused on purpose: this week is a
# reading order, not a re-derivation, so the frame has to be bit-identical to the audited one.
CACHE = Path(os.environ.get("TEMP", "."))/"opencode"/"ml09_cache"
CACHE.mkdir(parents=True, exist_ok=True)

def read_parquet(path, columns=None):
    with fs.open(path) as fh:
        return fastparquet.ParquetFile(fh).to_pandas(columns=columns)

def month_file(m):
    parts = sorted(p for p in fs.ls(FACT+"/month="+m, detail=False) if p.endswith(".parquet"))
    if not parts:
        raise FileNotFoundError(m)
    return parts[0]

def agg_month(m, my_clients):
    cached = CACHE/("agg_%s.parquet" % m)
    if cached.exists():
        g = pd.read_parquet(cached, engine=ENGINE)
        return g[g["client_hash_id"].isin(my_clients)].copy()
    t = read_parquet(month_file(m), DAILY_COLS)
    if my_clients:
        t = t[t["client_hash_id"].isin(my_clients)]
    t = t[t["gsc_data_available"].fillna(False).astype(bool)]
    t = t[t["gsc_impressions"] > 0]
    g = (t.groupby(KEYS, as_index=False)
           .agg(active_days=("report_date", "count"), impr=("gsc_impressions", "sum"),
                clicks=("gsc_clicks", "sum"), sum_pos=("gsc_sum_position", "sum")))
    assert g[KEYS].notna().all().all(), ("null join key in", m)
    assert not g.duplicated(KEYS).any(), ("duplicate join key in", m)
    g["month"] = m
    g.to_parquet(cached, engine=ENGINE)
    return g

clients = read_parquet(WAREHOUSE+"/dim_clients.parquet",
                       ["client_hash_id", "gsc_data_start", "has_gsc_access"])
gsc_start = pd.to_datetime(clients["gsc_data_start"].astype("string"))
MY_CLIENTS = set(clients.loc[(gsc_start <= FEATURE_START) & (clients["has_gsc_access"] != False),
                             "client_hash_id"])
monthly = {m: agg_month(m, MY_CLIENTS) for m in FEATURE_MONTHS + LABEL_MONTHS}

j  = monthly["2026-01"].rename(columns={"impr": "impr_jan", "clicks": "clicks_jan",
                                        "sum_pos": "sum_pos_jan", "active_days": "days_jan"})
b  = monthly["2026-02"].rename(columns={"impr": "impr_feb", "clicks": "clicks_feb",
                                        "sum_pos": "sum_pos_feb", "active_days": "days_feb"})
lb = monthly["2026-03"][KEYS+["impr"]].rename(columns={"impr": "impr_label"})

f = j.merge(b, on=KEYS, how="outer").merge(lb, on=KEYS, how="inner")

f["impr_prior"] = f["impr_jan"].fillna(0) + f["impr_feb"].fillna(0)
f["clicks_prior"] = f["clicks_jan"].fillna(0) + f["clicks_feb"].fillna(0)
f["sum_pos_prior"] = f["sum_pos_jan"].fillna(0) + f["sum_pos_feb"].fillna(0)
f["active_days_prior"] = f["days_jan"].fillna(0) + f["days_feb"].fillna(0)
f["ctr_prior_pct"] = np.where(f["impr_prior"] > 0, f["clicks_prior"]/f["impr_prior"]*100, np.nan)
f["wtd_pos_prior"] = np.where(f["impr_prior"] > 0, f["sum_pos_prior"]/f["impr_prior"], np.nan)
f["jan_rate_30d"] = f["impr_jan"].fillna(0)/DAYS_IN["2026-01"]*30
f["feb_rate_30d"] = f["impr_feb"].fillna(0)/DAYS_IN["2026-02"]*30
# Read, never scored: ML-09 measured this as a model feature at +0.0007 P@50 drop, i.e. nothing.
f["trend_pct_win"] = np.where(f["jan_rate_30d"] > 0,
                              (f["feb_rate_30d"]-f["jan_rate_30d"])/f["jan_rate_30d"]*100, np.nan)

content = read_parquet(WAREHOUSE+"/dim_content.parquet",
                       ["content_hash_id", "content_created_date", "content_updated_date", "word_count"])
for c in ("content_created_date", "content_updated_date"):
    content[c] = pd.to_datetime(content[c].astype("string"))
f = f.merge(content, on="content_hash_id", how="left")
f["content_age_days"] = (pd.Timestamp(DECISION_POINT) - f["content_created_date"]).dt.days

f["prior_rate_30d"] = f["impr_prior"]/PRIOR_DAYS*30
f["label_rate_30d"] = f["impr_label"]/LABEL_DAYS*30
f["drop_pct"] = np.where(f["prior_rate_30d"] > 0,
                         (f["label_rate_30d"] - f["prior_rate_30d"])/f["prior_rate_30d"]*100, np.nan)
f["in_pool"] = f["impr_prior"] >= POOL_FLOOR
f["is_material_decline"] = (f["in_pool"] & (f["prior_rate_30d"] >= VOLUME_FLOOR)
                            & (f["drop_pct"] <= DROP_FLOOR)).astype(int)

P = f[f["in_pool"]].copy()
P["log_impr_prior"] = np.log1p(P["impr_prior"])

# Same asserts as ML-09, in the same order. If the frame moved, this week is not comparable.
assert len(P) == 67279, len(P)
assert int(P["is_material_decline"].sum()) == 7445
assert P["client_hash_id"].nunique() == 24
assert float(P["is_material_decline"].mean()) == 0.11065860075209204
print("pool %s rows / %d clients / %s material declines / base %.4f   <- identical to ML-09"
      % ("{:,}".format(len(P)), P["client_hash_id"].nunique(),
         "{:,}".format(int(P["is_material_decline"].sum())), P["is_material_decline"].mean()))
print("windows: features %s -> %s | label %s -> %s | April and later never read"
      % (FEATURE_START, FEATURE_END, LABEL_START, LABEL_END))

# ---- THE model: ML-09's fitter, ML-09's five features, ML-09's nested-selected penalty. One fit. ----
FEATS5 = ["log_impr_prior", "ctr_prior_pct", "wtd_pos_prior", "active_days_prior", "content_age_days"]
L2_FINAL = 1.0                      # from w06_validation_metrics.json before_after_p50 l2_nested
y = P["is_material_decline"].to_numpy()

def build_design(A_raw, med, mu, sd, miss_cols):
    A = A_raw.astype(float).copy()
    for jj in miss_cols:
        col = A[:, jj]
        col[np.isnan(col)] = med[jj]
    Z = (A - mu)/sd
    parts = [Z, Z**2]
    n = Z.shape[1]
    for i in range(n):
        for k in range(i+1, n):
            parts.append((Z[:, i]*Z[:, k]).reshape(-1, 1))
    for jj in miss_cols:
        parts.append(np.isnan(A_raw[:, jj]).astype(float).reshape(-1, 1))
    Z2 = np.hstack(parts)
    return np.hstack([np.ones((Z2.shape[0], 1)), Z2])

def _sig(z):
    return 1.0/(1.0 + np.exp(-np.clip(z, -30, 30)))

N_FITS = []
def train_logreg(Z, yy, l2):
    N_FITS.append(1)               # asserted to be exactly 1 further down
    yy = yy.astype(float)
    n = len(yy); npos = yy.sum(); nneg = n - npos
    sw = np.where(yy == 1, n/(2.0*npos), n/(2.0*nneg))
    def obj(w):
        p = _sig(Z@w); eps = 1e-12
        nll = -np.sum(sw*(yy*np.log(p+eps) + (1-yy)*np.log(1-p+eps)))/n
        return nll + 0.5*l2*np.sum(w[1:]**2)
    def grad(w):
        p = _sig(Z@w)
        g = Z.T@(sw*(p-yy))/n
        g[1:] += l2*w[1:]
        return g
    res = minimize(obj, np.zeros(Z.shape[1]), jac=grad, method="L-BFGS-B", options={"maxiter": 500})
    return res.x, float(res.fun)

D = P[FEATS5].to_numpy(float)
med = np.nanmedian(D, axis=0)
miss = np.where(np.isnan(D).any(axis=0))[0]
mu = np.nanmean(D, axis=0); sd_ = np.nanstd(D, axis=0); sd_[sd_ == 0] = 1.0
W, NLL = train_logreg(build_design(D, med, mu, sd_, miss), y, L2_FINAL)
assert len(N_FITS) == 1, len(N_FITS)     # one deployment fit, no tuning, no probe model
print()
print("deployment model: 5 features, degree-2 logistic (incl. squares + pairwise), L2 = %.1f, fitted once on all %s pool rows"
      % (L2_FINAL, "{:,}".format(len(P))))
print("features: %s" % ", ".join(FEATS5))
print("excluded: trend_pct_win (ML-09 probe, +0.0007 P@50 drop) | ML-08's tuned L2 0.0001 (tuned on its own holdout)")
print("final training NLL %.6f" % NLL)

Q = P.copy()
Q["score"] = build_design(D, med, mu, sd_, miss) @ W          # log-odds
Q["p_decline"] = _sig(Q["score"])

# ---- reason codes + actions: only two real facts, so only two facts get named ----
BANDS = [-np.inf, VOLUME_FLOOR, 2000.0, np.inf]
BAND_NAMES = ["C_under_500", "B_500_to_2000", "A_over_2000"]
Q["volume_band"] = pd.cut(Q["prior_rate_30d"], BANDS, labels=BAND_NAMES).astype(str)
Q["is_sliding"] = (Q["trend_pct_win"] <= DOWN_FLOOR).fillna(False)
Q["reason_code"] = Q["volume_band"] + np.where(Q["is_sliding"], "_sliding", "_holding")

ACT_N = 50                          # ~one afternoon; also the depth ML-09 measured (P@50)
Q = Q.sort_values(["score", "wtd_pos_prior", "impr_prior", "content_hash_id"],
                  ascending=[False, True, False, True]).reset_index(drop=True)
Q["rank"] = np.arange(1, len(Q)+1)
Q["action"] = np.where(Q["rank"] <= ACT_N,
                       np.where(Q["is_sliding"], "refresh", "protect"),
                       "monitor")

QUEUE = REPO/"work"/"outputs"/"w07_action_queue.csv"
QOUT = Q[["rank", "action", "reason_code", "score", "p_decline", "volume_band", "is_sliding",
          "prior_rate_30d", "impr_prior", "wtd_pos_prior", "ctr_prior_pct",
          "active_days_prior", "content_age_days", "trend_pct_win",
          "client_hash_id", "content_hash_id"]].copy()
QOUT.to_csv(QUEUE, index=False)
print()
print("queue written: work/outputs/w07_action_queue.csv (%s rows x %d cols, %.1f MB)"
      % ("{:,}".format(len(QOUT)), QOUT.shape[1], QUEUE.stat().st_size/1e6))

print()
print("the %d rows above the cut:" % ACT_N)
SHOW = ["rank", "action", "reason_code", "p_decline", "prior_rate_30d", "wtd_pos_prior", "trend_pct_win"]
print(Q[SHOW].head(ACT_N).round(3).to_string(index=False))


pool 67,279 rows / 24 clients / 7,445 material declines / base 0.1107   <- identical to ML-09
windows: features 2026-01-01 -> 2026-02-28 | label 2026-03-01 -> 2026-03-31 | April and later never read

deployment model: 5 features, degree-2 logistic (incl. squares + pairwise), L2 = 1.0, fitted once on all 67,279 pool rows
features: log_impr_prior, ctr_prior_pct, wtd_pos_prior, active_days_prior, content_age_days
excluded: trend_pct_win (ML-09 probe, +0.0007 P@50 drop) | ML-08's tuned L2 0.0001 (tuned on its own holdout)
final training NLL 0.657118



queue written: work/outputs/w07_action_queue.csv (67,279 rows x 16 cols, 15.2 MB)

the 50 rows above the cut:
 rank  action         reason_code  p_decline  prior_rate_30d  wtd_pos_prior  trend_pct_win
    1 refresh A_over_2000_sliding      0.730      177927.458          5.739        -89.603
    2 protect A_over_2000_holding      0.729       186803.39          3.220        -10.573
    3 protect A_over_2000_holding      0.721      134367.458          2.513         -8.107
    4 protect A_over_2000_holding      0.721      115766.441          0.342        827.755
    5 protect A_over_2000_holding      0.718      106055.593          0.157       1368.573
    6 protect A_over_2000_holding      0.709       142763.39          5.947         -5.445
    7 protect A_over_2000_holding      0.708      131928.305          5.895          0.892
    8 refresh A_over_2000_sliding      0.706       97049.492          5.944        -98.814
    9 protect A_over_2000_holding      0.704      103013.898          

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

DRAFT

One person with one afternoon. They open pages in this order and decide what to do about them with
their own judgement. The queue replaces "which of 67,279 pages do I even look at", nothing more.

**The number that travels with this queue is ML-09's 0.300, not this week's.** That is Precision@50
measured out-of-fold across six client-grouped folds — about 15 of the top 50 were real material
declines, against 5.5 if you picked 50 at random. Per-fold it ranged 0.260 to 0.540 with a standard
deviation of 0.098, which is wide enough that I would not promise any single week's list will hit
0.300. I am reporting the model's fitted-on-this-pool number too, but only labelled as in-sample,
because a model scored on its own training rows flatters itself and ML-09 measured exactly that gap.

**The pool is not "pages at risk".** It is every `(client, content)` pair with 200+ prior
impressions — 67,279 rows, of which 7,445 declined. A row being in the queue means "large enough
page worth a look", not "about to lose traffic". The output is a reading order over a filtered
population, and the filter is inherited from ML-04, not chosen by me.

**Two ways this queue is narrower than it looks.** 40.2% of pool rows sit under the label's 500
volume floor and are therefore zero by construction — the model correctly pushes them down, but it
is agreeing with a rule the label already contained. And the queue's top is entirely the largest
pages, while the middle band holds the majority of real declines and never appears in it. Both
numbers are printed by the cell below rather than asserted here.

**The honest ceiling on volume.** Here is the part I did not expect to write. The label itself
requires `prior_rate_30d >= 500`, so any pool row under 500 impressions per 30 days *cannot* be a
positive — it is zero by construction, before March is even looked at. The model's one validated
input is exactly that volume. So a large share of what the ranking learned is a rule already
written into the label. I am not calling the model useless for it; a content team genuinely does
care more about a page earning 23,870 impressions a month (the pool's 99th percentile) than one
earning 154. But it means
the queue's skill is concentrated where the label made it easy, and the drop from 0.720 (ML-07's
frozen rule) to 0.300 (this model, honestly scored) is at least partly that, not only the model
being worse.

**One forward window.** Features end 2026-02-28, the label is March, and I never read April. ML-07's
own forward probe — scored against a label that cannot see January — fell to 0.340. If anyone asks
whether this generalises to next month, the true answer is that I have no evidence either way, and
the only honest way to get it is to run the next one.


In [2]:
# ---- limits, each one measured rather than asserted ----
M09 = json.load(open(REPO/"work"/"outputs"/"w06_validation_metrics.json", encoding="utf-8"))
BA = M09["before_after_p50"]["A contract (5 feats)"]
BB = M09["before_after_p50"]["B probe (+trend_pct_win)"]

def prec_at_k(scores, labels, k):
    return float(np.asarray(labels)[np.argsort(-np.asarray(scores), kind="stable")[:k]].mean())

s = Q["score"].to_numpy()
lab = Q["is_material_decline"].to_numpy()
base = float(lab.mean())
pak = {k: prec_at_k(s, lab, k) for k in (20, 50, 200)}

print("=" * 84)
print("IN-SAMPLE, this week's fit scored on the %s rows it was fitted on -- NOT a validation number" % "{:,}".format(len(Q)))
print("  P@20 %.3f | P@50 %.3f | P@200 %.3f | base %.4f" % (pak[20], pak[50], pak[200], base))
print("  ML-09 measured the same in-sample-vs-honest gap on the identical model: %.3f in-sample -> %.3f out-of-fold."
      % (BA["before_full_pool_mixed_in_sample"], BA["after_out_of_fold"]))
print()
print("THE NUMBER THIS QUEUE INHERITS (ML-09, out-of-fold, 6 client-grouped folds, 3 inner folds for L2)")
print("  contract 5 features : P@50 %.3f  (P@20 %.3f, P@200 %.3f)  per-fold %s  sd %.3f"
      % (BA["after_out_of_fold"], BA["after_out_of_fold_p20"], BA["after_out_of_fold_p200"],
         BA["per_fold_p50"], float(np.std(BA["per_fold_p50"], ddof=1))))
print("  probe +trend_pct_win: P@50 %.3f  -> adding the slope made the honest number worse, so it is out"
      % BB["after_out_of_fold"])
print("  ML-07 frozen rule    : P@50 %.3f on this same label, forward probe %.3f at base %.3f"
      % (M09["ml07_reference"]["p_at_50"], M09["ml07_reference"]["forward_p_at_50"],
         M07_BASE := json.load(open(REPO/"work"/"outputs"/"baseline_metrics.json",
                                    encoding="utf-8"))["forward_label_base_rate"]))
print("  chance at depth 50   : %.1f real declines from %d rows at base %.4f"
      % (50*base, len(Q), base))
print("  random floor at depth 50: %.3f (ML-07's floor, the number a queue has to beat)" % M09["ml07_reference"]["random_floor_p_at_50"])

vq = Q["prior_rate_30d"].quantile([0.10, 0.99])
print()
print("  pool prior_rate_30d spread, for the 'big pages matter more' intuition in the prose above:")
print("    p10 %.0f | median %.0f | p99 %.0f | max %.0f impressions per 30 days"
      % (vq.loc[0.10], Q["prior_rate_30d"].median(), vq.loc[0.99], Q["prior_rate_30d"].max()))

print()
print("=" * 84)
print("THE LABEL'S OWN VOLUME FLOOR -- how much of the ranking was already written into the target")
VOL_FLOOR_POS = int(((Q["prior_rate_30d"] >= VOLUME_FLOOR)).sum())
UNDER = Q[Q["prior_rate_30d"] < VOLUME_FLOOR]
print("  is_material_decline requires prior_rate_30d >= %d. That floor is in the LABEL, not the model." % VOLUME_FLOOR)
print("  pool rows under the floor : %s of %s (%.1f%%) -- all of them are 0 by construction, before March is read"
      % ("{:,}".format(len(UNDER)), "{:,}".format(len(Q)), 100.0*len(UNDER)/len(Q)))
print("  their measured material-decline rate: %.4f" % UNDER["is_material_decline"].mean())
print("  pool rows at or above the floor   : %s (%.1f%%), base rate %.4f"
      % ("{:,}".format(VOL_FLOOR_POS), 100.0*VOL_FLOOR_POS/len(Q),
         Q.loc[Q["prior_rate_30d"] >= VOLUME_FLOOR, "is_material_decline"].mean()))
print("  -> the model's only validated input (log_impr_prior) is the same quantity that floor is written on.")
print("     Part of the 0.300 is the model agreeing with a rule the label already contained.")
print()
print("  where the positives sit, by the same volume bands the reason codes use:")
BANDSUM = (Q.groupby("volume_band", observed=True)
             .agg(rows=("score", "size"), declines=("is_material_decline", "sum"),
                  base=("is_material_decline", "mean"), med_score=("score", "median"),
                  top50=("rank", lambda r: int((r <= ACT_N).sum())))
             .assign(share_of_positives=lambda d: d["declines"]/Q["is_material_decline"].sum()))
print(BANDSUM.round(4).to_string())

print()
print("=" * 84)
print("THE FINDING THAT MATTERS MOST FOR A QUEUE: this model is very close to a volume ranking")
VOL_ONLY = {"score": pak[50], "prior_rate_30d": prec_at_k(Q["prior_rate_30d"], lab, 50),
            "log_impr_prior": prec_at_k(Q["log_impr_prior"], lab, 50)}
for k, v in VOL_ONLY.items():
    print("  P@50 ranking by %-18s %.3f" % (k, v))
print("  the model and a raw sort-by-volume tie on depth 50 in-sample (%.3f each); at depth 200 the model"
      % pak[50])
print("  edges volume (%.3f vs %.3f). The four non-volume features are in the fit, and ML-09 measured them"
      % (pak[200], prec_at_k(Q["prior_rate_30d"], lab, 200)))
print("  at +0.035 / +0.010 / +0.007 / -0.018. That is the whole story of this queue.")
rho_vol = float(pd.Series(Q["score"]).corr(Q["prior_rate_30d"], method="spearman"))
print("  spearman(score, prior_rate_30d) = %.3f  -> the score is largely a repackaging of volume" % rho_vol)
print()
TOP = Q.head(ACT_N)
print("  CONSEQUENCE, and it is not a small one:")
print("   all %d rows above the cut are band A: %s" % (ACT_N, TOP["volume_band"].value_counts().to_dict()))
print("   their prior_rate_30d runs %.0f to %.0f, and only %d pool rows are that large."
      % (TOP["prior_rate_30d"].min(), TOP["prior_rate_30d"].max(),
         int((Q["prior_rate_30d"] >= TOP["prior_rate_30d"].min()).sum())))
first_b = int(Q.index[Q["volume_band"] == "B_500_to_2000"][0]) + 1
print("   band B holds %.1f%% of ALL material declines (%s of %s) and gets ZERO slots in the top %d."
      % (100.0*BANDSUM.loc["B_500_to_2000", "share_of_positives"],
         "{:,}".format(int(BANDSUM.loc["B_500_to_2000", "declines"])), "{:,}".format(int(y.sum())), ACT_N))
print("   the first band-B row appears at rank %s." % "{:,}".format(first_b))
print("   Best case INSIDE band B, its own best 50 score %.3f -- the same %.3f the whole-pool top 50 gets."
      % (prec_at_k(Q.loc[Q["volume_band"] == "B_500_to_2000", "score"], lab, 50), pak[50]))
print()
print("   So this queue is honest about the biggest pages and blind to the middle of the pool. I am not")
print("   going to fix it by re-weighting features on one label -- that is the move that produced ML-08's")
print("   0.0001 penalty. It needs a second forward window, which is exactly trigger 4 in section 4.")


IN-SAMPLE, this week's fit scored on the 67,279 rows it was fitted on -- NOT a validation number
  P@20 0.450 | P@50 0.360 | P@200 0.435 | base 0.1107
  ML-09 measured the same in-sample-vs-honest gap on the identical model: 0.480 in-sample -> 0.300 out-of-fold.

THE NUMBER THIS QUEUE INHERITS (ML-09, out-of-fold, 6 client-grouped folds, 3 inner folds for L2)
  contract 5 features : P@50 0.300  (P@20 0.350, P@200 0.385)  per-fold [0.26, 0.38, 0.54, 0.36, 0.48, 0.38]  sd 0.098
  probe +trend_pct_win: P@50 0.280  -> adding the slope made the honest number worse, so it is out
  ML-07 frozen rule    : P@50 0.720 on this same label, forward probe 0.340 at base 0.207
  chance at depth 50   : 5.5 real declines from 67279 rows at base 0.1107
  random floor at depth 50: 0.111 (ML-07's floor, the number a queue has to beat)

  pool prior_rate_30d spread, for the 'big pages matter more' intuition in the prose above:
    p10 154 | median 727 | p99 23870 | max 186803 impressions per 30 days

THE LA

  spearman(score, prior_rate_30d) = 0.802  -> the score is largely a repackaging of volume

  CONSEQUENCE, and it is not a small one:
   all 50 rows above the cut are band A: {'A_over_2000': 50}
   their prior_rate_30d runs 45067 to 186803, and only 191 pool rows are that large.
   band B holds 52.3% of ALL material declines (3,893 of 7,445) and gets ZERO slots in the top 50.
   the first band-B row appears at rank 9,741.
   Best case INSIDE band B, its own best 50 score 0.360 -- the same 0.360 the whole-pool top 50 gets.

   So this queue is honest about the biggest pages and blind to the middle of the pool. I am not
   going to fix it by re-weighting features on one label -- that is the move that produced ML-08's
   0.0001 penalty. It needs a second forward window, which is exactly trigger 4 in section 4.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

DRAFT

Every row leaves this notebook as a URL a person has to open, look at, and form an opinion about.
The model never touches the page. Nothing here publishes, unpublishes, rewrites a title, or sends a
message to a client, and I would not build that on a 0.300 ranking even if someone asked.

**What the reviewer has to check, per page.** That the page still exists and still earns roughly
what the row says — the frame is a March snapshot and the warehouse moves. That the reason code is
what they would have said anyway from Search Console, which is the real test of whether this queue
saved them time or just gave them a tidier list. And that the "material decline" in the row is a
traffic problem rather than a tracking problem: I filtered on days GSC reported data, so a page
that lost its data connection is not a page whose content failed.

**The no-go list, asserted in code below rather than promised in prose.**

- No automated edit, publish, unpublish, redirect, or client notification. Ever, off this score.
- No causal claim. The score says a page looks like a decline, not why, and not that a change will
  fix it. I ran no intervention, so I have no basis for "refreshing this page will recover X".
- No per-client reporting off this queue. 24 clients, one shared model, and 0.300 pooled — I have no
  per-client number, and a client-specific claim would be invented.
- No expanding the pool or the floor. `expand` and `prune` stay out until something tests them.
- No forecast past March. One forward window is not a trend.

The one that needs a human most: `refresh` on a sliding page is a judgement about whether the page
is worth saving, and that is a content call. If the page is sliding because it is deliberately
retiring, `refresh` is the wrong word and the model cannot tell the difference — it never sees
intent, only impressions.

**The check I would most want a reviewer to push on.** If they notice they are working through the
largest pages in the pool and nothing else, they are right. `A_over_2000` means "this page is
large", not "this page is at risk" — the reason codes cannot promise more than the model measured.
The cell below prints the band composition of the queue so this is visible rather than something
the reviewer has to discover.


In [3]:
# ---- the no-go list, checked rather than promised ----
print("=" * 84)
print("1. THE SCORE CANNOT SEE THE LABEL")
FORBIDDEN = ["impr_label", "label_rate_30d", "drop_pct", "is_material_decline",
             "log_impr_label_leak"]
used = FEATS5 + ["score", "p_decline", "is_sliding", "volume_band"]
bad = [c for c in FORBIDDEN if c in used]
for c in FEATS5:
    src = P[c]
    assert not np.isnan(src[P[c].notna()]).any()
print("   score inputs : %s" % ", ".join(FEATS5))
print("   label columns: %s" % ", ".join(FORBIDDEN))
print("   overlap      : %s" % (bad or "none -- the score cannot see the label"))
assert not bad
# the one column a reader could confuse for a leak: trend_pct_win is Jan->Feb only, and it is
# NOT an input. ML-09's leak trap added March impressions and hit P@50 1.000; I keep that distance.
print("   leak-trap reference from ML-09: honest P@50 %.3f -> %.3f with log_impr_label_leak added (%.1fx)"
      % (M09["leak_trap"]["honest_p50"], M09["leak_trap"]["leaked_p50"], M09["leak_trap"]["lift"]))
print("   trend_pct_win is read for the sliding flag only, never scored: ML-09 measured %+.4f P@50 drop from it"
      % M09["permutation_importance_p50_drop"]["B probe (+trend_pct_win)"]["trend_pct_win"])

print()
print("=" * 84)
print("2. WHICH REASON CODES ARE EVIDENCED, AND WHICH WOULD BE DECORATIVE")
DROPS = M09["permutation_importance_p50_drop"]["A contract (5 feats)"]
print("   ML-09 shuffled each feature inside held-out client blocks (20 reps) and measured the P@50 cost:")
for k, v in sorted(DROPS.items(), key=lambda kv: -kv[1]):
    tag = "carries the ranking" if v > 0.05 else ("noise" if v > 0 else "noise (scrambling HELPED)")
    print("     %-20s %+0.4f   %s" % (k, v, tag))
evidenced = [k for k, v in DROPS.items() if v > 0.05]
print("   evidenced inputs (%d): %s" % (len(evidenced), ", ".join(evidenced)))
print("   my reason codes name only prior-volume band + sliding, so they lean on %s and nothing else."
      % evidenced[0])
print("   a code naming ctr_prior_pct, content_age_days or active_days_prior would be decorative: those")
print("   measured %+.4f / %+.4f / %+.4f." % (DROPS["ctr_prior_pct"], DROPS["content_age_days"],
                                              DROPS["active_days_prior"]))

print()
print("=" * 84)
print("3. THE QUEUE, AS IT LEAVES THIS NOTEBOOK")
print("   rows %s | actions used %s | actions deliberately NOT used: expand, prune (ML-07 dropped both, untested here)"
      % ("{:,}".format(len(Q)), sorted(Q["action"].unique())))
print("   reason codes emitted (%d): %s" % (Q["reason_code"].nunique(), ", ".join(sorted(Q["reason_code"].unique()))))
print("   expand/prune appear anywhere in this notebook:",
      any("expand" in str(v) or "prune" in str(v) for v in
          [Q["action"].unique().tolist(), Q["reason_code"].unique().tolist()]))
ACT = (Q.groupby("action", observed=True)
         .agg(n=("score", "size"), med_p=("p_decline", "median"),
              declines=("is_material_decline", "sum")))
ACT["measured_decline_rate"] = ACT["declines"]/ACT["n"]
print()
print("   action mix, with the rate each action actually carries (in-sample, descriptive):")
print(ACT.round(4).to_string())
print()
print("   refresh/protect split the top %d by the sliding flag alone; monitor is everything below it." % ACT_N)
print("   No action here means 'edit this page'. It means 'look at this page first'.")


1. THE SCORE CANNOT SEE THE LABEL
   score inputs : log_impr_prior, ctr_prior_pct, wtd_pos_prior, active_days_prior, content_age_days
   label columns: impr_label, label_rate_30d, drop_pct, is_material_decline, log_impr_label_leak
   overlap      : none -- the score cannot see the label
   leak-trap reference from ML-09: honest P@50 0.300 -> 1.000 with log_impr_label_leak added (3.3x)
   trend_pct_win is read for the sliding flag only, never scored: ML-09 measured +0.0007 P@50 drop from it

2. WHICH REASON CODES ARE EVIDENCED, AND WHICH WOULD BE DECORATIVE
   ML-09 shuffled each feature inside held-out client blocks (20 reps) and measured the P@50 cost:
     log_impr_prior       +0.2343   carries the ranking
     wtd_pos_prior        +0.0352   noise
     content_age_days     +0.0097   noise
     active_days_prior    +0.0068   noise
     ctr_prior_pct        -0.0180   noise (scrambling HELPED)
   evidenced inputs (1): log_impr_prior
   my reason codes name only prior-volume band + slidi

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

DRAFT

I want each trigger to be a number I could be held to, so these come from what was actually
measured rather than from round numbers that feel safe.

- **The ranking stops beating the floor.** ML-09's honest P@50 is 0.300 against a random floor of
  0.111. If a future out-of-fold run lands at or below 0.200 I would stop shipping the queue and
  go back to the frame, because at that point it is doing less than "sort by volume".
- **Fold spread widens.** Per-fold P@50 ran 0.260 to 0.540, sd 0.098. If the next run's spread
  exceeds roughly 0.15 the model is being carried by a few clients, and a single client's queue
  would look better than the pooled number supports.
- **The pool moves.** 67,279 rows / 24 clients / 7,445 positives is asserted in this notebook. Any
  change means the upstream frame shifted, not that the model improved. ML-09 already caught an
  8-row / 1-client drift between the frame as built and what ML-04 recorded, with the pool
  unaffected — small enough to miss, exactly why the asserts exist.
- **Prior volume stops being the whole story.** The permutation evidence says one input does the
  work. If a later run shows CTR, age or active-days moving the needle at all, that is worth a
  look, because it would mean the frame changed rather than the model getting smarter.
- **The label floor moves.** `prior_rate_30d >= 500` and `drop_pct <= -20` are ML-04's constants.
  Changing either silently re-defines the target and invalidates the 0.300. I would want that
  change versioned and re-validated from scratch, not folded into a refresh.

**The one that is not a trigger: drift in my own prose.** If ML-07's rule or ML-09's audit gets
revised, this queue inherits the revision automatically — it reads those receipts rather than
hard-coding their numbers. It does *not* inherit a changed contract. If `FEATS5` ever changes,
this notebook's single fit is no longer the audited model and needs ML-09's validation run again
before the queue means anything.


In [4]:
# ---- triggers, each one a number that could fail ----
pf = np.array(BA["per_fold_p50"], float)
pf_sd = float(np.std(pf, ddof=1))
print("=" * 84)
print("TRIGGERS, with the current measured value next to each threshold")
print("  %-34s %-22s %s" % ("trigger", "measured now", "fire when"))
print("  %-34s %-22s %s" % ("out-of-fold P@50", "%.3f (ML-09, 6 folds)" % BA["after_out_of_fold"], "<= 0.200"))
print("  %-34s %-22s %s" % ("per-fold P@50 spread", "%.3f .. %.3f (sd %.3f)" % (pf.min(), pf.max(), pf_sd), "sd > 0.150"))
print("  %-34s %-22s %s" % ("pool rows / clients", "%s / %d" % ("{:,}".format(len(P)), P["client_hash_id"].nunique()), "any change"))
print("  %-34s %-22s %s" % ("positives / base rate", "%s / %.4f" % ("{:,}".format(int(y.sum())), base), "any change"))
print("  %-34s %-22s %s" % ("label volume floor", "prior_rate_30d >= %d" % VOLUME_FLOOR, "changed"))
print("  %-34s %-22s %s" % ("label drop floor", "drop_pct <= %.0f" % DROP_FLOOR, "changed"))
print("  %-34s %-22s %s" % ("forward windows validated", "1 (March only)", ">= 2 before forecasting"))
print("  %-34s %-22s %s" % ("features with real effect", "%d of %d" % (len(evidenced), len(FEATS5)), "not 1 -> re-audit"))

print()
print("=" * 84)
print("WHAT I AM DELIBERATELY NOT TRIGGERING ON")
print("  score movement week to week. The per-fold spread (%.3f) is larger than any drift I could detect" % pf_sd)
print("  at this sample size, so a queue that reorders itself between runs is noise, not news.")
print("  This week's own in-sample P@50 is %.3f and ML-09's out-of-fold is %.3f -- a %.3f gap from the SAME fit."
      % (pak[50], BA["after_out_of_fold"], pak[50]-BA["after_out_of_fold"]))
print("  Any monitoring that tracked the in-sample number would have 'improved' by %.0f%% and learned nothing."
      % (100.0*(pak[50]/BA["after_out_of_fold"]-1.0)))

print()
print("=" * 84)
print("SEED AND FIT COUNT, so a re-run is comparable")
print("  ML-09 seed %s | this fit is deterministic (L-BFGS-B from zeros, no sampling in the fit path)" % M09["seed"])
print("  fits performed in this notebook: %d -- no grid search, no probe model, no re-tuning" % len(N_FITS))
assert len(N_FITS) == 1
assert L2_FINAL == BA["l2_nested"], (L2_FINAL, BA["l2_nested"])
print("  L2 asserted equal to ML-09's nested-selected value: %.1f == %.1f" % (L2_FINAL, BA["l2_nested"]))


TRIGGERS, with the current measured value next to each threshold
  trigger                            measured now           fire when
  out-of-fold P@50                   0.300 (ML-09, 6 folds) <= 0.200
  per-fold P@50 spread               0.260 .. 0.540 (sd 0.098) sd > 0.150
  pool rows / clients                67,279 / 24            any change
  positives / base rate              7,445 / 0.1107         any change
  label volume floor                 prior_rate_30d >= 500  changed
  label drop floor                   drop_pct <= -20        changed
  forward windows validated          1 (March only)         >= 2 before forecasting
  features with real effect          1 of 5                 not 1 -> re-audit

WHAT I AM DELIBERATELY NOT TRIGGERING ON
  score movement week to week. The per-fold spread (0.098) is larger than any drift I could detect
  at this sample size, so a queue that reorders itself between runs is noise, not news.
  This week's own in-sample P@50 is 0.360 and ML-09's

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

DRAFT

Two artefacts, and they are deliberately different in weight.

**`work/outputs/w07_action_queue.csv`** — the full queue, 67,279 rows, one per `(client, content)`
pair, sorted by rank. Columns are the rank, action, reason code, log-odds score and probability, the
volume band and sliding flag, the five feature values in their raw units so a reader can check the
model's input without re-running anything, and the hashed ids. It is **not committed** — it is a
client-scoped extract and `work/**/*.csv` is gitignored. The cell above regenerates it in about a
second from the cached aggregates, and it is the file the paper's method section describes rather
than one it ships.

**`work/outputs/w07_action_playbook_metrics.json`** — the receipt. Every number quoted in the prose
above is a key here, so a claim in the write-up can be checked against a machine-readable value
instead of a screenshot of a notebook. This one is small, contains no client names or URLs, and is
safe to commit.

One thing I am deliberately not exporting: a per-client breakdown. With 24 clients and a pooled
0.300, any per-client number I printed would be noise with a client's name on it.


In [5]:
# ---- receipt + one figure. No client names, no URLs, no query text anywhere. ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGDIR = REPO/"work"/"figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
RECEIPT = REPO/"work"/"outputs"/"w07_action_playbook_metrics.json"

BANDSUM_F = {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
             for k, v in BANDSUM.to_dict("index").items()}

receipt = {
    "assignment": "ML-10",
    "audited_notebook": "work/notebooks/w07_action_playbook.ipynb",
    "upstream_audit": "work/notebooks/w06_validation_audit.ipynb",
    "role": "turns ML-09's audited model into a review queue; adds no new model",
    "decision_point": DECISION_POINT,
    "feature_window": [FEATURE_START, FEATURE_END],
    "label_window": [LABEL_START, LABEL_END],
    "april_or_later_read": False,
    "pool_rows": int(len(Q)),
    "pool_clients": int(Q["client_hash_id"].nunique()),
    "positives": int(int(y.sum())),
    "base_rate": base,
    "model": {
        "features": FEATS5,
        "excluded_probe_feature": "trend_pct_win",
        "excluded_probe_p50_drop": M09["permutation_importance_p50_drop"]["B probe (+trend_pct_win)"]["trend_pct_win"],
        "form": "degree-2 logistic regression (squares + all pairwise terms), L2",
        "l2": L2_FINAL,
        "l2_provenance": "w06_validation_metrics.json before_after_p50['A contract (5 feats)'].l2_nested",
        "l2_rejected": {"ml08_shipped_tuned_on_its_own_holdout": M09["ml08_as_shipped"]["best_l2"]},
        "fits_performed": int(len(N_FITS)),
        "fitted_on": "entire pool (deployment fit, in-sample by construction)",
        "training_nll": float(NLL),
    },
    "p_at_k_this_fit_in_sample": {("P@%d" % k): pak[k] for k in (20, 50, 200)},
    "p_at_k_inherited_from_ml09_out_of_fold": {
        "P@20": BA["after_out_of_fold_p20"], "P@50": BA["after_out_of_fold"],
        "P@200": BA["after_out_of_fold_p200"],
        "per_fold_P@50": BA["per_fold_p50"],
        "per_fold_sd": float(np.std(BA["per_fold_p50"], ddof=1)),
        "random_floor": M09["ml07_reference"]["random_floor_p_at_50"],
    },
    "queue": {
        "cut_depth": ACT_N,
        "cut_rationale": "one afternoon of pages; the depth ML-09 measured (P@50)",
        "score_is_near_a_volume_ranking": {
            "p_at_50_by_model_score_in_sample": pak[50],
            "p_at_50_by_prior_rate_30d_in_sample": prec_at_k(Q["prior_rate_30d"], lab, 50),
            "p_at_50_by_log_impr_prior_in_sample": prec_at_k(Q["log_impr_prior"], lab, 50),
            "p_at_200_by_model_score_in_sample": pak[200],
            "p_at_200_by_prior_rate_30d_in_sample": prec_at_k(Q["prior_rate_30d"], lab, 200),
            "spearman_score_vs_prior_rate_30d": float(pd.Series(Q["score"]).corr(Q["prior_rate_30d"], method="spearman")),
            "top_cut_all_one_band": True,
            "top_cut_min_prior_rate_30d": float(Q.head(ACT_N)["prior_rate_30d"].min()),
            "pool_rows_at_or_above_that_volume": int((Q["prior_rate_30d"] >= Q.head(ACT_N)["prior_rate_30d"].min()).sum()),
            "band_B_share_of_all_positives": float(BANDSUM.loc["B_500_to_2000", "share_of_positives"]),
            "band_B_rows_in_top_cut": int(BANDSUM.loc["B_500_to_2000", "top50"]),
            "band_B_first_rank": int(Q.index[Q["volume_band"] == "B_500_to_2000"][0]) + 1,
            "band_B_best_case_p_at_50": prec_at_k(Q.loc[Q["volume_band"] == "B_500_to_2000", "score"], lab, 50),
            "reading": "model ties a raw volume sort at depth 50; the top cut is the largest pages only, "
                       "while the middle band holds the majority of declines and never appears in it",
            "not_fixed_because": "re-weighting features on a single label is what produced ML-08's holdout-tuned penalty",
        },
        "sort": "score desc, then wtd_pos_prior asc, impr_prior desc, content_hash_id asc",
        "deterministic": True,
        "actions_used": sorted(Q["action"].unique().tolist()),
        "actions_dropped_and_why": {
            "expand": "ML-07 dropped it: the word-count idea came from thin_visible_page and was never tested",
            "prune": "ML-07 dropped it: nothing in evidence says a page earning 200+ prior impressions should be removed",
        },
        "action_counts": {k: int(v) for k, v in Q["action"].value_counts().items()},
        "action_measured_decline_rate_in_sample": {k: float(v) for k, v in ACT["measured_decline_rate"].items()},
        "reason_codes": sorted(Q["reason_code"].unique().tolist()),
        "reason_code_names_only": ["prior-volume band", "Jan->Feb sliding flag (reported, not scored)"],
    },
    "label_volume_floor_caveat": {
        "floor": "prior_rate_30d >= %d" % VOLUME_FLOOR,
        "pool_rows_under_floor": int(len(UNDER)),
        "share_of_pool_under_floor": float(len(UNDER)/len(Q)),
        "measured_rate_under_floor": float(UNDER["is_material_decline"].mean()),
        "pool_rows_at_or_above_floor": int(VOL_FLOOR_POS),
        "base_rate_at_or_above_floor": float(Q.loc[Q["prior_rate_30d"] >= VOLUME_FLOOR, "is_material_decline"].mean()),
        "reading": "the model's only validated input is the same quantity the label floor is written on",
        "volume_band_summary": BANDSUM_F,
    },
    "evidenced_vs_decorative": {
        "permutation_p50_drop_ml09": DROPS,
        "evidenced_inputs": evidenced,
        "reason_codes_avoid_unevidenced_features": True,
    },
    "no_go_list": [
        "no automated edit, publish, unpublish, redirect or client notification off this score",
        "no causal claim: no intervention was run, so no 'fixing this recovers X'",
        "no per-client reporting: 24 clients, one pooled model, no per-client number measured",
        "expand and prune stay out until something tests them",
        "no forecast past March: one forward window",
    ],
    "monitoring_triggers": {
        "out_of_fold_p_at_50_fire_at_or_below": 0.200,
        "out_of_fold_p_at_50_measured": BA["after_out_of_fold"],
        "per_fold_sd_fire_above": 0.150,
        "per_fold_sd_measured": float(np.std(BA["per_fold_p50"], ddof=1)),
        "pool_rows": int(len(Q)), "pool_clients": int(Q["client_hash_id"].nunique()),
        "positives": int(int(y.sum())), "base_rate": base,
        "label_volume_floor": VOLUME_FLOOR, "label_drop_floor": DROP_FLOOR,
        "forward_windows_validated": 1,
        "deliberately_not_a_trigger": "week-to-week score movement; per-fold spread exceeds detectable drift",
    },
    "withheld": [
        "no second forward window (April never read)",
        "no re-weighting to surface the middle volume band: that needs a second window, not a tuned fit",
        "no per-client breakdown: no per-client number was measured",
        "no new validation: the queue inherits ML-09's out-of-fold numbers rather than producing its own",
        "no claim that the model beats ML-07 on March ranking: both sit on the same blended label",
    ],
    "exports": {
        "queue_csv": "work/outputs/w07_action_queue.csv",
        "queue_csv_rows": int(len(Q)),
        "queue_csv_committed": False,
        "queue_csv_gitignore_reason": "work/**/*.csv",
        "receipt_json": "work/outputs/w07_action_playbook_metrics.json",
        "receipt_json_committed": True,
        "figure_png": "work/figures/w07_decline_rate_by_rank.png",
    },
    "client_names_or_urls_present": False,
    "csv_committed": False,
}
RECEIPT.write_text(json.dumps(receipt, indent=2), encoding="utf-8")

# ---- one figure: is the queue's top actually denser in declines than its bottom? ----
N_BUCKET = 40
q = Q.copy()
q["bucket"] = pd.cut(q["rank"], bins=np.linspace(0, len(q), N_BUCKET+1).astype(int), labels=False, include_lowest=True)
buck = q.groupby("bucket", observed=True).agg(
    n=("is_material_decline", "size"),
    rate=("is_material_decline", "mean"),
    rank_lo=("rank", "min"), rank_hi=("rank", "max"))
fig, ax = plt.subplots(figsize=(8.6, 4.4))
ax.plot(buck["rank_lo"], buck["rate"], color="#2b6cb0", lw=1.7, marker="o", ms=3.4,
        label="measured material-decline rate")
ax.axhline(base, color="#c53030", lw=1.3, ls="--",
           label="pool base rate %.4f" % base)
ax.set_xlabel("queue rank (1 = open first, %s = last)" % "{:,}".format(len(q)))
ax.set_ylabel("material-decline rate within rank bucket")
ax.set_title("The reading order beats the pool base rate where it matters (in-sample, 1 label window)", fontsize=10.5)
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
FIG = FIGDIR/"w07_decline_rate_by_rank.png"
fig.savefig(FIG, dpi=150)
plt.close(fig)

print("receipt written: work/outputs/w07_action_playbook_metrics.json (%d bytes)" % RECEIPT.stat().st_size)
print("figure written : work/figures/w07_decline_rate_by_rank.png (%.1f KB)" % (FIG.stat().st_size/1e3))
print()
print("=" * 84)
print("EXPORT CHECK")
for pth, committed in [(QUEUE, "no  (work/**/*.csv is gitignored, client-scoped extract)"),
                       (RECEIPT, "yes (small, no client names or URLs)"),
                       (FIG, "yes (small PNG)")]:
    print("  %-52s %9d bytes  committed: %s"
          % (pth.relative_to(REPO).as_posix(), pth.stat().st_size, committed))
    assert pth.exists() and pth.stat().st_size > 0
print()
print("top 5 of the queue, the way the reviewer sees it:")
TOP5 = ["rank", "action", "reason_code", "p_decline", "prior_rate_30d", "wtd_pos_prior"]
print(Q[TOP5].head(5).round(4).to_string(index=False))
print()
print("bottom 3, for contrast -- what 'monitor' looks like at the far end:")
print(Q[TOP5].tail(3).round(4).to_string(index=False))


receipt written: work/outputs/w07_action_playbook_metrics.json (7155 bytes)
figure written : work/figures/w07_decline_rate_by_rank.png (67.7 KB)

EXPORT CHECK
  work/outputs/w07_action_queue.csv                     15155643 bytes  committed: no  (work/**/*.csv is gitignored, client-scoped extract)
  work/outputs/w07_action_playbook_metrics.json             7155 bytes  committed: yes (small, no client names or URLs)
  work/figures/w07_decline_rate_by_rank.png                67713 bytes  committed: yes (small PNG)

top 5 of the queue, the way the reviewer sees it:
 rank  action         reason_code  p_decline  prior_rate_30d  wtd_pos_prior
    1 refresh A_over_2000_sliding     0.7303     177927.4576         5.7386
    2 protect A_over_2000_holding     0.7288     186803.3898         3.2201
    3 protect A_over_2000_holding     0.7212     134367.4576         2.5129
    4 protect A_over_2000_holding     0.7212     115766.4407         0.3417
    5 protect A_over_2000_holding     0.7184     10

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] I re-used ML-09's model rather than fitting a new one (one fit, L2 = 1.0, FEATS5 only)
- [ ] Every action label came from ML-07's vocabulary; `expand` and `prune` are not in my queue
- [ ] Every reason code names something ML-09 measured, not something I assumed
- [ ] I read the ML-09 audit directly instead of trusting my memory of it
- [ ] The queue CSV is regenerated, not committed; the receipt and figure are
- [ ] I did not touch `scripts/`, and `work/` only grew
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
